> **Note (repository):** the training cell was set to `epochs=40` to match the arguments stored inside the released `bdd100k_7model.pt` (40 epochs, patience 10, `optimizer=auto` -> AdamW, batch 16, 640 px). The saved output below comes from an earlier, interrupted session that was configured for 50 epochs and stopped at epoch 33; it is kept unchanged as a record of that attempt and is not the run that produced the released model. See README.


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [1]:
import torch
print(f"Is GPU available? {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU Name: {torch.cuda.get_device_name(0)}")
else:
    print("Running on CPU (Please go to Runtime > Change runtime type)")

Is GPU available? True
GPU Name: Tesla T4


In [ ]:
%pip install ultralytics -q
import os
import shutil
import zipfile
from ultralytics import YOLO
from google.colab import drive

# SETUP & CONFIGURATION
print("Mounting Google Drive...")
drive.mount('/content/drive')
DRIVE_ZIP_PATH = '/content/drive/MyDrive/bdd100k_7class.zip'
RUNTIME_DIR = '/content/bdd_workspace'

# CLEAN EXTRACT
print(f"Extracting dataset to {RUNTIME_DIR}...")
if os.path.exists(RUNTIME_DIR):
    shutil.rmtree(RUNTIME_DIR) # Clean start
os.makedirs(RUNTIME_DIR)

# Unzip
with zipfile.ZipFile(DRIVE_ZIP_PATH, 'r') as zip_ref:
    zip_ref.extractall(RUNTIME_DIR)

# AUTO-LOCATE DATASET & FIX CONFIG
print("Locating data.yaml and fixing paths...")

# Find the folder containing data.yaml
dataset_root = None
yaml_path = None

for root, dirs, files in os.walk(RUNTIME_DIR):
    if 'data.yaml' in files:
        dataset_root = root
        yaml_path = os.path.join(root, 'data.yaml')
        break

if not dataset_root:
    print("CRITICAL ERROR: Could not find 'data.yaml' in the zip file.")
    exit()

print(f" Found dataset at: {dataset_root}")

# Read the original names from the file
with open(yaml_path, 'r') as f:
    old_content = f.read()

# Extract the class names block (names: [...])
names_block = ""
if "names:" in old_content:
    names_block = old_content[old_content.find("names:"):]
elif "nc:" in old_content:
    names_block = old_content[old_content.find("nc:"):]

# Write the NEW config with the CORRECT absolute path
new_yaml_content = f"""
path: {dataset_root}
train: train/images
val: valid/images
test: test/images

{names_block}
"""

with open(yaml_path, 'w') as f:
    f.write(new_yaml_content)

print(f"Config updated! YOLO will use the {names_block.count(',') + 1} classes found in the file.")

# TRAINING
print("\n Starting Training...")
model = YOLO('yolov8n.pt')

results = model.train(
    data=yaml_path,
    epochs=40,
    imgsz=640,
    batch=16,        # Safe batch size for T4 GPU
    project="blind_stick_project",
    name="bdd100k_7run",
    device=0,
    verbose=True,
    patience=10
)

# TRAINING EXPORT
save_path = '/content/drive/MyDrive/bdd100k_7model.pt'
print(f"\n Saving model to {save_path}...")
shutil.copy(f"{results.save_dir}/weights/best.pt", save_path)
print(" DONE.")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 35.0 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Mounting Google Drive...
Mounted at /content/drive
Extracting dataset to /content/bdd_workspace...
Locating data.yaml and fixing paths...
 Found dataset at: /content/bdd_workspace
Config updated! YOLO will use the 7 classes found in the file.

 Starting Training...
Ultralytics 8.3.248 🚀 Python-3.12.12 torch-2.9.0+cu126 CUDA:0 (Tesla T4, 15095MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=